# Tutorial: Databricks Connect Smoke Test

Run this notebook from VS Code with the project `.venv` kernel. It verifies that local notebook cells can authenticate through the saved Databricks CLI OAuth profile, execute Spark work on the `research-compute` cluster, display a returned table, and save a figure that Claude/Codex can inspect from disk.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
from databricks.connect import DatabricksSession
from databricks.sdk.core import Config
from pyspark.sql import functions as F

PROFILE = "hindman.gmail.com@auth.researchaccelerator.org"
CLUSTER_ID = "0303-193859-1ff54asc"

config = Config(profile=PROFILE, cluster_id=CLUSTER_ID)
spark = DatabricksSession.builder.sdkConfig(config).getOrCreate()
spark

## Authentication Check

This should return your Databricks workspace user and a current timestamp. If it fails here, the issue is authentication, cluster access, or cluster state rather than VS Code itself.

In [ ]:
identity = spark.sql("select current_user() as user, current_timestamp() as ts").toPandas()
identity

## Spark Table Check

The Spark work runs on Databricks compute. The resulting Spark DataFrame is collected back as pandas so VS Code can display it as a normal notebook table.

In [ ]:
pdf = spark.range(12).withColumn("square", F.col("id") * F.col("id")).toPandas()
pdf

## Figure Check

Inline notebook figures are useful for you, but AI coding agents are most reliable when the figure is saved to a file path. This cell writes a PNG under `figs/`.

In [ ]:
figs_dir = Path("figs")
figs_dir.mkdir(exist_ok=True)
out = figs_dir / "databricks_connect_smoke.png"

ax = pdf.plot(x="id", y="square", marker="o", legend=False)
ax.set_title("Databricks Connect Smoke Test")
ax.set_xlabel("id")
ax.set_ylabel("id squared")
ax.figure.tight_layout()
ax.figure.savefig(out, dpi=150)
plt.close(ax.figure)

out